# 6. MCP client: discover, then govern

Model Context Protocol standardizes how clients discover and invoke server capabilities. It does not decide whether a capability is trusted or authorized. We first use an offline protocol-shaped client, then optionally connect to the instructor’s local stdio server.

## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Discover an MCP tool, classify it locally, authorize it through harness policy, and invoke it through mock or stdio transport.

Architecture reference: [Day 5 diagrams D17](../../diagrams/source/day_05.md).

### Expected observation

course_lookup is discovered, classified read-only, and called only after local validation and policy. Exact IDs, timing, and live wording will vary.

## Concept briefing

## MCP: protocol, not permission

Model Context Protocol lets a client initialise a session, discover server capabilities
and invoke them through a common contract. A server may expose tools, resources or prompts.
The protocol improves interoperability; it does not establish trust.

An MCP tool description and its results are untrusted external content. Before importing
a discovered tool, the harness should consider server origin, schema, local risk,
permitted agents, arguments, timeout, output handling and logging. A server changing its
advertised tools must not silently expand application authority.

The Day 5 rule is therefore:

```text
discovery is not authorization
```

The client discovers the tool, the harness classifies it, local policy authorises or
pauses it, and only then does the protocol call occur.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
import asyncio
async def offline_demo():
    client=FakeMCPClient(); tools=await client.list_tools()
    print("Discovered:",tools)
    raw=tools[0]
    spec=ToolSpec(raw["name"],raw["description"],raw["inputSchema"],"read")
    cfg=AgentConfig("mcp_demo","Use one supplied fact.",[spec.name])
    print("Local policy:",decide(cfg,spec))
    if decide(cfg,spec)=="allow": print("Result:",await client.call_tool(spec.name,{"topic":"mcp"}))
await offline_demo()

## Official SDK local lab

Install the instructor-pinned stable SDK (`mcp[cli]`). The supplied server is course infrastructure; students need not write it. `StdioMCPClient` uses `StdioServerParameters`, `stdio_client`, `ClientSession.initialize()`, `list_tools()`, and `call_tool()`. On Windows use the Python executable that launches the current environment.

In [ ]:
# Set RUN_REAL_MCP=1 before launching Jupyter after installing the pinned SDK.
import importlib.util,os,sys
if os.getenv("RUN_REAL_MCP")=="1" and importlib.util.find_spec("mcp"):
    client=StdioMCPClient(sys.executable,[str(DAY/"instructor_mcp_server.py")])
    tools,result=await client.list_and_optionally_call("course_lookup",{"topic":"harness"})
    print([tool.name for tool in tools]); print(result)
else:
    print("Real MCP skipped. Complete the offline policy path above or enable RUN_REAL_MCP=1.")

## Security checkpoint

Before importing an MCP tool into the registry, inspect server origin, tool description/schema, local risk classification, allowed agents, arguments, output handling, timeout, and logging. A server can change its advertised tools; rediscovery is not automatic authorization.

## Your turn

Change its local risk to external and show that discovery stays identical while authorization changes.

## Recap

MCP standardizes capability exchange; the harness retains trust and permission decisions. Name one responsibility that deliberately remains application-specific.